# 조합 5 프롬프트 모드

PR ④ 파이프라인(복원 포함)으로 조합 5 를 레포 코드로 검증하고, 프롬프트 반영도를 높일
레버를 본다. 이전 어휘 실험(normal_01·02·09, seed 42)과 같은 조건에서 시작한다.

레포 feat/face-restore. 조합 3 은 올리지 않는다.

In [ ]:
%cd /content
import importlib
import os
import shutil
import sys
import time
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
from PIL import Image

drive.mount("/content/drive")

REPO = "/content/SalonCutAI"
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q -b feat/fullres-recompose https://github.com/qja0707/SalonCutAI.git {REPO}
!pip install -q diffusers==0.39.0 transformers==5.14.1 peft==0.19.1 accelerate==1.14.0 \
    insightface onnxruntime mediapipe==1.0.0 opencv-contrib-python-headless \
    facexlib torchvision

os.environ["IMAGE_GEN_ENABLED"] = "1"
os.environ["SALON_STORAGE_DIR"] = "/content/storage"
sys.path.insert(0, f"{REPO}/backend")
importlib.invalidate_caches()

from src.ai_engine.image_gen import downloads, loader, pipeline, settings, storage

downloads.ensure_models()
print("missing:", downloads.missing_files())

loader.get_face_app()
loader.get_landmarker()
loader.get_segmenter()
loader.get_codeformer()
loader.get_face_helper()
loader.get_combo5()

import torch
print(f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 기준선

normal_01·02·09 × 동물상 3종(강아지·고양이·여우), 무표정, seed 42.

In [ ]:
NORMAL = Path("/content/drive/MyDrive/saloncut_data/test_images/normal")
OUT5 = Path("/content/drive/MyDrive/saloncut_data/outputs/combo5_0825")
OUT5.mkdir(parents=True, exist_ok=True)
SEED = 42

N_TARGETS = {
    "normal_01_short_dark": ("여성", "20대"),
    "normal_02_long_dark": ("여성", "20대"),
    "normal_09_dark_skin": ("남성", "20대"),
}
STYLES = {"puppy": "강아지상", "cat": "고양이상", "fox": "여우상"}


def opts5(gender, age, face_style):
    return SimpleNamespace(
        face=SimpleNamespace(
            mode="prompt",
            prompt=SimpleNamespace(
                ethnicity="한국인", gender=gender, age=age,
                face_style=face_style, expression="무표정", skin_tone="", makeup="",
            ),
        )
    )


srcs5 = {}
for name, (gender, age) in N_TARGETS.items():
    src_path = next(NORMAL.glob(f"{name}.*"))
    src = storage.to_stored_size(Image.open(src_path).convert("RGB"))
    srcs5[name] = src
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src)
    axes[0].set_title(f"{name[7:9]} src", fontsize=13)

    for ax, (key, style) in zip(axes[1:], STYLES.items()):
        t = time.time()
        fin = pipeline._run_prompt_mode(src, opts5(gender, age, style), SEED)
        fin.save(OUT5 / f"c5_{name}_{key}.png")
        ax.imshow(fin)
        ax.set_title(f"{name[7:9]} {key}", fontsize=13)
        print(f"{name}  {key}  {time.time() - t:.0f}s")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_baseline.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
SALON = Path("/content/drive/MyDrive/saloncut_data/test_images/salon")
S_TARGETS = {
    "salon_01_long_wave_brown": ("여성", "20대"),
    "salon_04_long_wave_black": ("여성", "20대"),
}

for name, (gender, age) in S_TARGETS.items():
    src = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
    srcs5[name] = src
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src)
    axes[0].set_title(f"{name[6:8]} src", fontsize=13)

    for ax, (key, style) in zip(axes[1:], STYLES.items()):
        t = time.time()
        fin = pipeline._run_prompt_mode(src, opts5(gender, age, style), SEED)
        fin.save(OUT5 / f"c5_{name}_{key}.png")
        ax.imshow(fin)
        ax.set_title(f"{name[6:8]} {key}", fontsize=13)
        print(f"{name}  {key}  {time.time() - t:.0f}s")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_baseline.png", dpi=100, bbox_inches="tight")
    plt.show()

N_TARGETS.update(S_TARGETS)  # 이후 스윕은 5장으로

## 레버 1 — guidance

7.5(기준) → 10 → 13. 텍스트 조건을 세게 따르게 한다. 과포화·왜곡이 나오는 지점까지.

In [ ]:
GUIDANCES = [10.0, 13.0]


def short(name):
    return name.split("_")[1]


for name, (gender, age) in N_TARGETS.items():
    src = srcs5[name]
    fig, axes = plt.subplots(len(GUIDANCES) + 1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7 * (len(GUIDANCES) + 1)))

    axes[0, 0].imshow(src)
    axes[0, 0].set_title(f"{short(name)} src", fontsize=13)
    for ax, key in zip(axes[0, 1:], STYLES):
        ax.imshow(Image.open(OUT5 / f"c5_{name}_{key}.png"))
        ax.set_title(f"g7.5 {key}", fontsize=13)

    for row, g in enumerate(GUIDANCES, start=1):
        settings.COMBO5_GUIDANCE = g
        for ax, (key, style) in zip(axes[row, 1:], STYLES.items()):
            t = time.time()
            fin = pipeline._run_prompt_mode(src, opts5(gender, age, style), SEED)
            fin.save(OUT5 / f"c5_{name}_{key}_g{g}.png")
            ax.imshow(fin)
            ax.set_title(f"g{g} {key}", fontsize=13)
            print(f"{name}  g{g}  {key}  {time.time() - t:.0f}s")

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_guidance.png", dpi=100, bbox_inches="tight")
    plt.show()

settings.COMBO5_GUIDANCE = 7.5

## 레버 2 — 얼굴 크롭 인페인팅

얼굴 박스 pad 1.6 정사각 크롭 → 1024 인페인팅 → 되붙임. latent 얼굴 폭 2.4배.
guidance 7.5 고정. 후처리는 기존 _postprocess.

In [ ]:
import cv2
import torch
from src.ai_engine.image_gen import masks, prompt_map


def combo5_crop_generate(img, options, seed, pad=1.6):
    """얼굴 주변만 잘라 1024 로 인페인팅한 뒤 원본 자리에 되붙인다."""
    face_mask = masks.build_face_mask(img)
    if face_mask is None:
        raise ValueError("얼굴 마스크를 만들 수 없다")
    hair_mask = masks.build_hair_mask(img)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    faces = loader.get_face_app().get(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR))
    f = max(faces, key=lambda f: f.bbox[2] - f.bbox[0])
    x1, y1, x2, y2 = f.bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * pad
    W, H = img.size
    box = (
        int(max(0, cx - side / 2)), int(max(0, cy - side / 2)),
        int(min(W, cx + side / 2)), int(min(H, cy + side / 2)),
    )

    crop = img.crop(box)
    crop_mask = gen_mask.crop(box)
    crop_r, mask_r = masks.resize_for_sdxl(crop, crop_mask)

    prompt = prompt_map.build_face_prompt(options)
    out = loader.get_combo5()(
        prompt=prompt,
        negative_prompt=settings.FACE_NEGATIVE,
        image=crop_r,
        mask_image=mask_r,
        strength=settings.COMBO5_STRENGTH,
        num_inference_steps=settings.COMBO5_STEPS,
        guidance_scale=settings.COMBO5_GUIDANCE,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]

    full = img.copy()
    full.paste(out.resize(crop.size, Image.LANCZOS), box[:2])
    return full, face_mask, hair_mask, box


def run_crop(img, opts, seed):
    full, face_mask, hair_mask, box = combo5_crop_generate(img, opts.face.prompt, seed)
    return pipeline._postprocess(img, full, face_mask, hair_mask), box


for name, (gender, age) in N_TARGETS.items():
    src = srcs5[name]
    fig, axes = plt.subplots(2, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 14))
    axes[0, 0].imshow(src)
    axes[0, 0].set_title(f"{short(name)} src", fontsize=13)
    axes[1, 0].axis("off")
    for ax, key in zip(axes[0, 1:], STYLES):
        ax.imshow(Image.open(OUT5 / f"c5_{name}_{key}.png"))
        ax.set_title(f"base {key}", fontsize=13)

    for ax, (key, style) in zip(axes[1, 1:], STYLES.items()):
        t = time.time()
        fin, box = run_crop(src, opts5(gender, age, style), SEED)
        fin.save(OUT5 / f"c5_{name}_{key}_crop.png")
        ax.imshow(fin)
        ax.set_title(f"crop {key}", fontsize=13)
        print(f"{name}  crop {key}  {time.time() - t:.0f}s  box {box}")

    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_crop.png", dpi=100, bbox_inches="tight")
    plt.show()

## 레버 3 — GPT-Image 편집

SDXL 이 동물상 개념을 모르는 게 원인이면 개념을 아는 모델로 바꿔본다.
얼굴 크롭(pad 1.6, 1024×1024) + gen_mask 알파 마스크 + 서비스 프롬프트 그대로 → images.edit → 되붙임 → 기존 후처리.
1장 × 3종만. 사진 외부 전송이라 팀 합의 전 실험용.

In [ ]:
import base64
import io
from getpass import getpass

from openai import OpenAI

client = OpenAI(api_key=getpass("OpenAI API key: "))
GPT_MODEL = "gpt-image-2"  # 안 되면 gpt-image-1


def gpt_crop_edit(img, options, pad=1.6):
    face_mask = masks.build_face_mask(img)
    hair_mask = masks.build_hair_mask(img)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    faces = loader.get_face_app().get(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR))
    f = max(faces, key=lambda f: f.bbox[2] - f.bbox[0])
    x1, y1, x2, y2 = f.bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * pad
    W, H = img.size
    box = (int(max(0, cx - side / 2)), int(max(0, cy - side / 2)),
           int(min(W, cx + side / 2)), int(min(H, cy + side / 2)))

    crop = img.crop(box).resize((1024, 1024), Image.LANCZOS)
    m = gen_mask.crop(box).resize((1024, 1024), Image.NEAREST)
    # 편집할 곳(얼굴)을 투명으로
    alpha = Image.fromarray(255 - np.array(m.convert("L")))
    rgba = crop.copy().convert("RGBA")
    rgba.putalpha(alpha)

    img_buf, mask_buf = io.BytesIO(), io.BytesIO()
    crop.save(img_buf, "PNG"); img_buf.seek(0); img_buf.name = "image.png"
    rgba.save(mask_buf, "PNG"); mask_buf.seek(0); mask_buf.name = "mask.png"

    prompt = prompt_map.build_face_prompt(options)
    prompt += ", same head pose and gaze as the original, photorealistic, natural skin"

    res = client.images.edit(model=GPT_MODEL, image=img_buf, mask=mask_buf,
                             prompt=prompt, size="1024x1024", n=1)
    out = Image.open(io.BytesIO(base64.b64decode(res.data[0].b64_json))).convert("RGB")

    full = img.copy()
    full.paste(out.resize((box[2] - box[0], box[3] - box[1]), Image.LANCZOS), box[:2])
    return full, face_mask, hair_mask, out


name = "salon_01_long_wave_brown"
gender, age = N_TARGETS[name]
src = srcs5[name]

fig, axes = plt.subplots(2, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 14))
axes[0, 0].imshow(src); axes[0, 0].set_title(f"{short(name)} src", fontsize=13)
axes[1, 0].axis("off")
for col, (key, style) in enumerate(STYLES.items(), start=1):
    t = time.time()
    full, fm, hm, raw = gpt_crop_edit(src, opts5(gender, age, style).face.prompt)
    fin = pipeline._postprocess(src, full, fm, hm)
    fin.save(OUT5 / f"c5_{name}_{key}_gpt.png")
    raw.save(OUT5 / f"c5_{name}_{key}_gpt_raw.png")
    axes[0, col].imshow(raw); axes[0, col].set_title(f"gpt raw {key}", fontsize=13)
    axes[1, col].imshow(fin); axes[1, col].set_title(f"gpt final {key}", fontsize=13)
    print(f"{name}  gpt {key}  {time.time() - t:.0f}s")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_gpt.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
for name in ["normal_09_dark_skin", "salon_04_long_wave_black", "normal_01_short_dark"]:
    gender, age = N_TARGETS[name]
    src = srcs5[name]
    fig, axes = plt.subplots(2, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 14))
    axes[0, 0].imshow(src); axes[0, 0].set_title(f"{short(name)} src", fontsize=13)
    axes[1, 0].axis("off")
    for col, (key, style) in enumerate(STYLES.items(), start=1):
        t = time.time()
        full, fm, hm, raw = gpt_crop_edit(src, opts5(gender, age, style).face.prompt)
        fin = pipeline._postprocess(src, full, fm, hm)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt.png")
        raw.save(OUT5 / f"c5_{name}_{key}_gpt_raw.png")
        axes[0, col].imshow(raw); axes[0, col].set_title(f"gpt raw {key}", fontsize=13)
        axes[1, col].imshow(fin); axes[1, col].set_title(f"gpt final {key}", fontsize=13)
        print(f"{name}  gpt {key}  {time.time() - t:.0f}s")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
from src.ai_engine.image_gen import compose


def face_box(img, pad=1.6):
    faces = loader.get_face_app().get(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR))
    f = max(faces, key=lambda f: f.bbox[2] - f.bbox[0])
    x1, y1, x2, y2 = f.bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * pad
    W, H = img.size
    return (int(max(0, cx - side / 2)), int(max(0, cy - side / 2)),
            int(min(W, cx + side / 2)), int(min(H, cy + side / 2)))


def pp_norestore(img, out, face_mask, hair_mask):
    out = out.resize(img.size, Image.LANCZOS)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)
    out = compose.color_transfer(out, img, gen_mask)
    comp = compose.recompose_with_hair(img, out, face_mask, hair_mask)
    return compose.keep_brows(img, comp)


def pp_minimal(img, out, face_mask, hair_mask):
    out = out.resize(img.size, Image.LANCZOS)
    comp = compose.recompose_with_hair(img, out, face_mask, hair_mask)
    return compose.keep_brows(img, comp)


VARIANTS = {"restore": pipeline._postprocess, "no restore": pp_norestore, "recompose only": pp_minimal}

for name in ["salon_01_long_wave_brown", "normal_09_dark_skin", "salon_04_long_wave_black", "normal_01_short_dark"]:
    src = srcs5[name]
    box = face_box(src)
    fm, hm = masks.build_face_mask(src), masks.build_hair_mask(src)

    fig, axes = plt.subplots(len(VARIANTS) + 1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7 * (len(VARIANTS) + 1)))
    axes[0, 0].imshow(src); axes[0, 0].set_title(f"{short(name)} src", fontsize=13)
    for col, key in enumerate(STYLES, start=1):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_raw.png")
        full = src.copy()
        full.paste(raw.resize((box[2] - box[0], box[3] - box[1]), Image.LANCZOS), box[:2])
        axes[0, col].imshow(raw); axes[0, col].set_title(f"raw {key}", fontsize=13)
        for row, (vname, fn) in enumerate(VARIANTS.items(), start=1):
            fin = fn(src, full, fm, hm)
            fin.save(OUT5 / f"c5_{name}_{key}_gpt_{vname.replace(' ', '_')}.png")
            axes[row, col].imshow(fin); axes[row, col].set_title(f"{vname} {key}", fontsize=12)
    for row in range(1, len(VARIANTS) + 1):
        axes[row, 0].axis("off")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_pp.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def pp_recompose(img, out, face_mask, hair_mask):
    out = out.resize(img.size, Image.LANCZOS)
    return compose.recompose_with_hair(img, out, face_mask, hair_mask)


for name in ["salon_01_long_wave_brown", "normal_09_dark_skin", "salon_04_long_wave_black", "normal_01_short_dark"]:
    src = srcs5[name]
    box = face_box(src)
    fm, hm = masks.build_face_mask(src), masks.build_hair_mask(src)
    x1, y1, x2, y2 = box
    fw = x2 - x1
    eye_box = (x1 + fw // 6, y1 + fw // 4, x2 - fw // 6, y1 + fw * 5 // 9)  # 눈썹~눈 아래

    fig, axes = plt.subplots(len(STYLES), 4, figsize=(28, 6 * len(STYLES)))
    for row, key in enumerate(STYLES):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_raw.png")
        full = src.copy()
        full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
        no_brow = pp_recompose(src, full, fm, hm)
        with_brow = compose.keep_brows(src, no_brow)
        no_brow.save(OUT5 / f"c5_{name}_{key}_gpt_recompose_nobrow.png")

        for ax, (lab, im) in zip(axes[row], [("src", src), ("raw pasted", full), ("recompose, no brow", no_brow), ("recompose + brow", with_brow)]):
            ax.imshow(im.crop(eye_box))
            ax.set_title(f"{short(name)} {key} — {lab}", fontsize=12)
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_brow.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
for name in ["normal_09_dark_skin", "normal_01_short_dark", "salon_04_long_wave_black"]:
    src = srcs5[name]
    box = face_box(src)
    x1, y1, x2, y2 = box
    fw = x2 - x1
    eye_box = (x1 + fw // 6, y1 + fw // 4, x2 - fw // 6, y1 + fw * 5 // 9)
    fm = masks.build_face_mask(src)
    hm_default = masks.build_hair_mask(src)
    hm_zero = masks.build_hair_mask(src, dilate=0)

    fig, axes = plt.subplots(len(STYLES), 3, figsize=(21, 6 * len(STYLES)))
    for row, key in enumerate(STYLES):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_raw.png")
        full = src.copy()
        full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
        d = compose.recompose_with_hair(src, full, fm, hm_default)
        z = compose.recompose_with_hair(src, full, fm, hm_zero)
        z.save(OUT5 / f"c5_{name}_{key}_gpt_recompose_dilate0.png")
        for ax, (lab, im) in zip(axes[row], [("raw pasted", full), ("hair dilate default", d), ("hair dilate 0", z)]):
            ax.imshow(im.crop(eye_box))
            ax.set_title(f"{short(name)} {key} — {lab}", fontsize=12)
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_dilate.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def sentence_prompt(gender, age, style):
    who = {"여성": "woman", "남성": "man"}[gender]
    ages = {"20대": "in her 20s" if gender == "여성" else "in his 20s"}.get(age, "")
    traits = {
        "강아지상": (
            "a distinctly puppy-like face: large round eyes with outer corners that droop gently downward, "
            "soft rounded upper eyelids, a small soft nose tip, full cheeks, and a gentle innocent expression"
        ),
        "고양이상": (
            "a distinctly cat-like face: long almond-shaped eyes with outer corners that tilt clearly upward, "
            "narrow eye height, a slim straight nose with a fine bridge, small well-defined lips, and a cool sharp expression"
        ),
        "여우상": (
            "a distinctly fox-like face: long narrow eyes with lowered lids and outer corners stretched upward, "
            "a thin straight nose, a slim pointed chin, and a sly elegant expression"
        ),
    }[style]
    return (
        f"Replace the face with a different Korean {who} {ages} who has {traits}. "
        "Keep the same head pose, gaze direction and lighting. Photorealistic, natural skin, no exaggerated makeup."
    )


def gpt_edit_crop(crop, rgba, prompt):
    ib, mb = io.BytesIO(), io.BytesIO()
    crop.save(ib, "PNG"); ib.seek(0); ib.name = "image.png"
    rgba.save(mb, "PNG"); mb.seek(0); mb.name = "mask.png"
    res = client.images.edit(model=GPT_MODEL, image=ib, mask=mb, prompt=prompt, size="1024x1024", n=1)
    return Image.open(io.BytesIO(base64.b64decode(res.data[0].b64_json))).convert("RGB")


name = "salon_01_long_wave_brown"
gender, age = N_TARGETS[name]
src = srcs5[name]
box = face_box(src)
fm, hm = masks.build_face_mask(src), masks.build_hair_mask(src)
gm = masks.build_gen_mask(fm, hm)
crop = src.crop(box).resize((1024, 1024), Image.LANCZOS)
m = gm.crop(box).resize((1024, 1024), Image.NEAREST)
rgba = crop.copy().convert("RGBA")
rgba.putalpha(Image.fromarray(255 - np.array(m.convert("L"))))

fig, axes = plt.subplots(2, len(STYLES) + 1, figsize=(7 * (len(STYLES) + 1), 14))
axes[0, 0].imshow(crop); axes[0, 0].set_title("src crop", fontsize=13)
axes[1, 0].axis("off")
for col, (key, style) in enumerate(STYLES.items(), start=1):
    tag = prompt_map.build_face_prompt(opts5(gender, age, style).face.prompt)
    tag += ", same head pose and gaze as the original, photorealistic, natural skin"
    sent = sentence_prompt(gender, age, style)
    for row, (label, prompt) in enumerate([("tag", tag), ("sentence", sent)]):
        t = time.time()
        raw = gpt_edit_crop(crop, rgba, prompt)
        raw.save(OUT5 / f"c5_{name}_{key}_gpt_raw_{label}.png")
        axes[row, col].imshow(raw); axes[row, col].set_title(f"{label} {key}", fontsize=13)
        print(f"{label}  {key}  {time.time() - t:.0f}s")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_gpt_tag_vs_sentence.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def sentence_prompt_v2(gender, age, style, expression="무표정"):
    who = {"여성": "woman", "남성": "man"}[gender]
    ages = {"20대": "in her 20s" if gender == "여성" else "in his 20s"}.get(age, "")
    traits = {
        "강아지상": "a puppy-like face: large round eyes with outer corners that droop gently downward, soft rounded upper eyelids, a small soft nose tip, and full cheeks",
        "고양이상": "a cat-like face: long almond-shaped eyes with outer corners that tilt upward, narrow eye height, a slim straight nose with a fine bridge, and small well-defined lips",
        "여우상": "a fox-like face: long narrow eyes with slightly lowered lids and outer corners stretched upward, a thin straight nose, and a slim pointed chin",
    }[style]
    expr = {"무표정": "a neutral relaxed expression with lips closed"}.get(expression, expression)
    return (
        f"Replace the face with a different Korean {who} {ages} who has {traits}, with {expr}. "
        "Keep the same head pose, gaze direction and lighting. Photorealistic, natural skin, no exaggerated makeup."
    )


def run_gpt_final(src, prompt):
    box = face_box(src)
    x1, y1, x2, y2 = box
    fm = masks.build_face_mask(src)
    hm_edit = masks.build_hair_mask(src)            # 편집 마스크용 (팽창 포함)
    hm_paste = masks.build_hair_mask(src, dilate=0)  # 되붙임용
    gm = masks.build_gen_mask(fm, hm_edit)
    crop = src.crop(box).resize((1024, 1024), Image.LANCZOS)
    m = gm.crop(box).resize((1024, 1024), Image.NEAREST)
    rgba = crop.copy().convert("RGBA")
    rgba.putalpha(Image.fromarray(255 - np.array(m.convert("L"))))
    raw = gpt_edit_crop(crop, rgba, prompt)
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    return compose.recompose_with_hair(src, full, fm, hm_paste), raw


for name in ["salon_01_long_wave_brown", "normal_09_dark_skin", "salon_04_long_wave_black", "normal_01_short_dark"]:
    gender, age = N_TARGETS[name]
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=13)
    for ax, (key, style) in zip(axes[1:], STYLES.items()):
        t = time.time()
        fin, raw = run_gpt_final(src, sentence_prompt_v2(gender, age, style))
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v2_final.png")
        raw.save(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
        ax.imshow(fin); ax.set_title(f"{key}", fontsize=13)
        print(f"{name}  {key}  {time.time() - t:.0f}s")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_v2_final.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def pp_alpha(img, out, face_mask, hair_mask0, alpha):
    out = out.resize(img.size, Image.LANCZOS)
    if alpha > 0:
        gen_mask = masks.build_gen_mask(face_mask, hair_mask0)
        out = compose.color_transfer(out, img, gen_mask, alpha=alpha)
    return compose.recompose_with_hair(img, out, face_mask, hair_mask0)


ALPHAS = [0.0, 0.3, 0.5, 1.0]
for name, key in [("normal_09_dark_skin", "fox"), ("normal_01_short_dark", "cat"), ("salon_01_long_wave_brown", "puppy")]:
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    fw = x2 - x1
    face_crop = (x1 - fw // 6, y1, x2 + fw // 6, y2 + fw // 3)  # 얼굴+목

    fig, axes = plt.subplots(1, len(ALPHAS) + 1, figsize=(6 * (len(ALPHAS) + 1), 7))
    axes[0].imshow(src.crop(face_crop)); axes[0].set_title(f"{short(name)} {key} src", fontsize=12)
    for ax, a in zip(axes[1:], ALPHAS):
        fin = pp_alpha(src, full, fm, hm0, a)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v2_alpha{a}.png")
        ax.imshow(fin.crop(face_crop)); ax.set_title(f"alpha {a}", fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_{key}_alpha.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
STYLE_SENT = {
    "강아지상": "a puppy-like face: large round eyes with outer corners that droop gently downward, soft rounded upper eyelids, a small soft nose tip, and full cheeks",
    "고양이상": "a cat-like face: long almond-shaped eyes with outer corners that tilt upward, narrow eye height, a slim straight nose with a fine bridge, and small well-defined lips",
    "여우상": "a fox-like face: long narrow eyes with slightly lowered lids and outer corners stretched upward, a thin straight nose, and a slim pointed chin",
}


def sentence_prompt_v3(o):
    """FacePromptOptions(또는 SimpleNamespace) 전 축을 문장으로. 표에 없는 값은 건너뛴다."""
    eth = prompt_map.ETHNICITY.get(o.ethnicity, "")
    gen = prompt_map.GENDER.get(o.gender, "person")
    age = prompt_map.AGE_GROUP.get(o.age, "")
    who = " ".join(p for p in [eth, gen, age] if p)

    traits = STYLE_SENT.get(o.face_style) or prompt_map.FACE_STYLE.get(o.face_style, "")
    expr = prompt_map.EXPRESSION.get(o.expression, "")
    skin = prompt_map.SKIN_TONE.get(o.skin_tone, "")
    mk = prompt_map.MAKEUP.get(o.makeup, "")

    s = f"Replace the face with a different {who}"
    if traits:
        s += f" who has {traits}"
    s += "."
    if expr:
        s += f" Expression: {expr}."
    if skin:
        s += f" Skin: {skin}."
    if mk:
        s += f" Makeup: {mk}."
    s += " Keep the same head pose, gaze direction and lighting. Photorealistic, natural skin."
    return s


def opts_full(gender, age, face_style, expression="무표정", skin_tone="", makeup=""):
    return SimpleNamespace(ethnicity="한국인", gender=gender, age=age, face_style=face_style,
                           expression=expression, skin_tone=skin_tone, makeup=makeup)


COMBOS = {
    "cat+smile+fair+cherry": opts_full("여성", "20대", "고양이상", "자연스러운 미소", "밝은 톤", "체리 레드"),
    "puppy+eyesmile+tan+dewy": opts_full("여성", "20대", "강아지상", "눈웃음", "태닝 톤", "물광"),
    "fox+haughty+natural+plum": opts_full("여성", "20대", "여우상", "도도한", "자연 톤", "다크 로맨틱"),
    "deer+neutral+fair+bare": opts_full("여성", "20대", "사슴상", "무표정", "밝은 톤", "노메이크업"),
}

name = "salon_01_long_wave_brown"
src = srcs5[name]
fig, axes = plt.subplots(1, len(COMBOS) + 1, figsize=(6 * (len(COMBOS) + 1), 7))
axes[0].imshow(src); axes[0].set_title("src", fontsize=12)
for ax, (label, o) in zip(axes[1:], COMBOS.items()):
    prompt = sentence_prompt_v3(o)
    print(label, "→", prompt, "\n")
    t = time.time()
    fin, raw = run_gpt_final(src, prompt)
    fin.save(OUT5 / f"c5_{name}_combo_{label}.png")
    raw.save(OUT5 / f"c5_{name}_combo_{label}_raw.png")
    ax.imshow(fin); ax.set_title(label, fontsize=11)
    print(f"{label}  {time.time() - t:.0f}s")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_combos.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def skin_target_mask(img, face_mask):
    """몸피부+얼굴피부 클래스에서 얼굴 윤곽 마스크를 뺀 영역. 목·귀·이마 띠."""
    cat = masks._category_mask(img)
    skin = Image.fromarray((np.isin(cat, [2, 3]).astype(np.uint8) * 255)).resize(img.size, Image.NEAREST)
    fm = np.array(face_mask.convert("L")) > 127
    out = np.array(skin.convert("L")).copy()
    out[fm] = 0
    return Image.fromarray(out)


def skin_match(img, ref_mask, tgt_mask, alpha=1.0):
    """tgt_mask 영역의 LAB 평균을 ref_mask 영역 평균으로 옮긴다. 분산은 원본 유지."""
    s = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2LAB).astype(np.float32)
    tm = np.array(tgt_mask.resize(img.size).convert("L")) > 127
    rm = np.array(ref_mask.resize(img.size).convert("L")) > 127
    out = s.copy()
    for c in range(3):
        s_mean, r_mean = s[..., c][tm].mean(), s[..., c][rm].mean()
        out[..., c][tm] = s[..., c][tm] + (r_mean - s_mean) * alpha
    return Image.fromarray(cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB))


tan_opts = opts_full("여성", "20대", "강아지상", "눈웃음", "태닝 톤", "물광")

for name in ["salon_01_long_wave_brown", "normal_01_short_dark"]:
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    gm = masks.build_gen_mask(fm, hm0)
    tgt = skin_target_mask(src, fm)

    raw_path = OUT5 / f"c5_{name}_combo_puppy+eyesmile+tan+dewy_raw.png"
    if raw_path.exists():
        raw = Image.open(raw_path)
    else:
        _, raw = run_gpt_final(src, sentence_prompt_v3(tan_opts))
        raw.save(raw_path)

    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    cur = compose.recompose_with_hair(src, full, fm, hm0)
    matched = skin_match(cur, gm, tgt)
    matched.save(OUT5 / f"c5_{name}_tan_skinmatched.png")

    overlay = Image.composite(Image.new("RGB", src.size, (255, 0, 0)), src, tgt.point(lambda v: 128 if v > 127 else 0))
    fw = x2 - x1
    crop_box = (max(0, x1 - fw), max(0, y1 - fw // 2), min(src.width, x2 + fw), min(src.height, y2 + fw))

    fig, axes = plt.subplots(1, 4, figsize=(28, 7))
    for ax, (lab, im) in zip(axes, [("src", src), ("skin target", overlay), ("face only tan", cur), ("skin matched", matched)]):
        ax.imshow(im.crop(crop_box)); ax.set_title(f"{short(name)} {lab}", fontsize=12); ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_skin_match.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
COMBOS2 = {
    "salon_01_long_wave_brown": {
        "rabbit+bigsmile+natural": opts_full("여성", "20대", "토끼상", "활짝 웃는", "", "꾸안꾸"),
        "bear+eyesmile+velvet": opts_full("여성", "20대", "곰상", "눈웃음", "", "벨벳 블러"),
        "edgy+stare+darkromantic": opts_full("여성", "20대", "개성 있는", "강렬한 응시", "", "다크 로맨틱"),
    },
    "normal_09_dark_skin": {
        "wolf+stare+mutedrose": opts_full("남성", "20대", "늑대상", "강렬한 응시", "", "말린 장미"),
    },
}

for name, combos in COMBOS2.items():
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(combos) + 1, figsize=(6 * (len(combos) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, (label, o) in zip(axes[1:], combos.items()):
        prompt = sentence_prompt_v3(o)
        print(label, "→", prompt, "\n")
        t = time.time()
        fin, raw = run_gpt_final(src, prompt)
        fin.save(OUT5 / f"c5_{name}_combo_{label}.png")
        raw.save(OUT5 / f"c5_{name}_combo_{label}_raw.png")
        ax.imshow(fin); ax.set_title(label, fontsize=11)
        print(f"{label}  {time.time() - t:.0f}s")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_combos2.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def dilate_mask(mask, px):
    if px <= 0:
        return mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (px * 2 + 1, px * 2 + 1))
    return Image.fromarray(cv2.dilate(np.array(mask.convert("L")), k))


RATIOS = [0.0, 0.06, 0.12]
for name, key in [("salon_01_long_wave_brown", "puppy"), ("normal_09_dark_skin", "cat"), ("salon_04_long_wave_black", "fox")]:
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)

    fig, axes = plt.subplots(1, len(RATIOS) + 1, figsize=(6 * (len(RATIOS) + 1), 7))
    axes[0].imshow(src.crop(jaw)); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, r in zip(axes[1:], RATIOS):
        fm_d = dilate_mask(fm, int(fw * r))
        fin = compose.recompose_with_hair(src, full, fm_d, hm0)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v2_facedil{r}.png")
        ax.imshow(fin.crop(jaw)); ax.set_title(f"face dilate {r}", fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_{key}_facedilate.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
name, key = "salon_01_long_wave_brown", "cat"
src = srcs5[name]
box = face_box(src)
x1, y1, x2, y2 = box
fw = x2 - x1
fm, hm_edit, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src), masks.build_hair_mask(src, dilate=0)
gm = masks.build_gen_mask(fm, hm_edit)

crop = src.crop(box).resize((1024, 1024), Image.LANCZOS)
m = gm.crop(box).resize((1024, 1024), Image.NEAREST)
mask_vis = Image.composite(Image.new("RGB", crop.size, (255, 0, 0)), crop, m.point(lambda v: 128 if v > 127 else 0))
raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
full = src.copy()
full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
full = compose.color_transfer(full, src, masks.build_gen_mask(fm, hm0), alpha=0.3)
fin = compose.recompose_with_hair(src, full, dilate_mask(fm, int(fw * 0.06)), hm0)

steps = [("1 원본", src), ("2 얼굴 크롭", crop), ("3 편집 마스크(빨강)", mask_vis), ("4 gpt 편집 결과", raw), ("5 되붙임+재합성", fin)]
fig, axes = plt.subplots(1, 5, figsize=(30, 7))
for ax, (lab, im) in zip(axes, steps):
    ax.imshow(im); ax.set_title(lab, fontsize=13); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_gpt_process.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def finalize_from_raw(src, raw, alpha=0.3, face_dilate=0.06):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm, hm0), alpha=alpha)
    return compose.recompose_with_hair(src, full, dilate_mask(fm, int(fw * face_dilate)), hm0)


for name in ["salon_01_long_wave_brown", "normal_09_dark_skin", "salon_04_long_wave_black", "normal_01_short_dark"]:
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=13)
    for ax, key in zip(axes[1:], STYLES):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
        fin = finalize_from_raw(src, raw)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v3_final.png")
        ax.imshow(fin); ax.set_title(key, fontsize=13)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_v3_final.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def union_face_mask(src, full):
    fm_src = masks.build_face_mask(src)
    fm_new = masks.build_face_mask(full)
    if fm_new is None:
        return fm_src
    u = np.maximum(np.array(fm_src.convert("L")), np.array(fm_new.convert("L")))
    return Image.fromarray(u)


for name, key in [("salon_01_long_wave_brown", "cat"), ("salon_01_long_wave_brown", "fox"), ("normal_09_dark_skin", "fox")]:
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    full = compose.color_transfer(full, src, masks.build_gen_mask(fm, hm0), alpha=0.3)
    jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)

    variants = {
        "dilate 0.06": dilate_mask(fm, int(fw * 0.06)),
        "dilate 0.10": dilate_mask(fm, int(fw * 0.10)),
        "union": union_face_mask(src, full),
        "union + 0.03": dilate_mask(union_face_mask(src, full), int(fw * 0.03)),
    }
    fig, axes = plt.subplots(1, len(variants) + 1, figsize=(6 * (len(variants) + 1), 7))
    axes[0].imshow(src.crop(jaw)); axes[0].set_title(f"{short(name)} {key} src", fontsize=12)
    for ax, (lab, m) in zip(axes[1:], variants.items()):
        fin = compose.recompose_with_hair(src, full, m, hm0)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v3_{lab.replace(' ', '').replace('+', '_')}.png")
        ax.imshow(fin.crop(jaw)); ax.set_title(lab, fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_{key}_facemask_union.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def finalize_from_raw_v4(src, raw, alpha=0.3, face_dilate=0.06):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    fm_d = dilate_mask(fm, int(fw * face_dilate))
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm_d, hm0), alpha=alpha)  # 같은 마스크
    return compose.recompose_with_hair(src, full, fm_d, hm0)


for name, key in [("salon_01_long_wave_brown", "cat"), ("salon_01_long_wave_brown", "fox"), ("normal_09_dark_skin", "fox")]:
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
    old = finalize_from_raw(src, raw)         # 셀 22 방식 (색 정합 = 원본 마스크)
    new = finalize_from_raw_v4(src, raw)      # 색 정합 = 팽창 마스크
    new.save(OUT5 / f"c5_{name}_{key}_gpt_v4_final.png")

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    for ax, (lab, im) in zip(axes, [("src", src), ("v3 (ct on orig mask)", old), ("v4 (ct on dilated mask)", new)]):
        ax.imshow(im.crop(jaw)); ax.set_title(f"{short(name)} {key} {lab}", fontsize=12); ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_{key}_ct_mask.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def finalize_v5(src, raw, alpha=0.3, face_dilate=0.06, blur=25):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    fm_d = dilate_mask(fm, int(fw * face_dilate))
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm_d, hm0), alpha=alpha)
    keep = settings.RECOMPOSE_BLUR
    settings.RECOMPOSE_BLUR = blur
    try:
        return compose.recompose_with_hair(src, full, fm_d, hm0)
    finally:
        settings.RECOMPOSE_BLUR = keep


name, key = "salon_01_long_wave_brown", "cat"
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)
raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")

DIL = [0.06, 0.12]
BLUR = [25, 10, 5]
fig, axes = plt.subplots(len(DIL), len(BLUR) + 1, figsize=(6 * (len(BLUR) + 1), 7 * len(DIL)))
for i, d in enumerate(DIL):
    axes[i, 0].imshow(src.crop(jaw)); axes[i, 0].set_title("src", fontsize=12)
    for j, b in enumerate(BLUR):
        fin = finalize_v5(src, raw, face_dilate=d, blur=b)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v5_d{d}_b{b}.png")
        axes[i, j + 1].imshow(fin.crop(jaw)); axes[i, j + 1].set_title(f"dilate {d}  blur {b}", fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_dilate_blur.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def run_gpt_final_v6(src, prompt, alpha=0.3, edit_dilate=0.10, face_dilate=0.06):
    box = face_box(src)
    x1, y1, x2, y2 = box
    fw = x2 - x1
    fm = masks.build_face_mask(src)
    hm_edit = masks.build_hair_mask(src)
    hm0 = masks.build_hair_mask(src, dilate=0)
    fm_edit = dilate_mask(fm, int(fw * edit_dilate))            # GPT 에 보내는 마스크는 넓게
    gm_edit = masks.build_gen_mask(fm_edit, hm_edit)
    crop = src.crop(box).resize((1024, 1024), Image.LANCZOS)
    m = gm_edit.crop(box).resize((1024, 1024), Image.NEAREST)
    rgba = crop.copy().convert("RGBA")
    rgba.putalpha(Image.fromarray(255 - np.array(m.convert("L"))))
    raw = gpt_edit_crop(crop, rgba, prompt)
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    fm_d = dilate_mask(fm, int(fw * face_dilate))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm_d, hm0), alpha=alpha)
    return compose.recompose_with_hair(src, full, fm_d, hm0), raw, full


name, key = "salon_01_long_wave_brown", "cat"
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)

raw_old = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v2_raw.png")
full_old = src.copy()
full_old.paste(raw_old.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))

fin_new, raw_new, full_new = run_gpt_final_v6(src, sentence_prompt_v2("여성", "20대", "고양이상"))
fin_new.save(OUT5 / f"c5_{name}_{key}_gpt_v6_final.png")
raw_new.save(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")

fig, axes = plt.subplots(1, 4, figsize=(24, 7))
for ax, (lab, im) in zip(axes, [("src", src), ("old raw pasted (edit mask = oval)", full_old), ("new raw pasted (edit mask +10%)", full_new), ("new final", fin_new)]):
    ax.imshow(im.crop(jaw)); ax.set_title(lab, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_editmask.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
KEEP = " Keep the face the same size and position as the original; do not shrink or shift it."

prompt_cat = sentence_prompt_v2("여성", "20대", "고양이상") + KEEP
results = {}
for d in [0.06, 0.10]:
    fin, raw, full = run_gpt_final_v6(src, prompt_cat, edit_dilate=d)
    fin.save(OUT5 / f"c5_{name}_{key}_gpt_v6_edit{d}.png")
    results[f"edit +{int(d*100)}%"] = fin
    print(f"edit dilate {d} done")

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
axes[0].imshow(src.crop(jaw)); axes[0].set_title("src", fontsize=12)
for ax, (lab, im) in zip(axes[1:], results.items()):
    ax.imshow(im.crop(jaw)); ax.set_title(lab, fontsize=12)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_editmask_size.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
STYLE_SENT = {
    "강아지상": "a puppy-like face: large round eyes with outer corners that droop gently downward, soft rounded upper eyelids, a small soft nose tip, and full cheeks",
    "고양이상": "a cat-like face: long almond-shaped eyes with outer corners that tilt upward, narrow eye height, a slim straight nose with a fine bridge, and small well-defined lips",
    "여우상": "a fox-like face: long narrow eyes with slightly lowered lids and outer corners stretched upward, a thin straight nose, and a slim pointed chin",
}


def sentence_prompt_v3(o):
    eth = prompt_map.ETHNICITY.get(o.ethnicity, "")
    gen = prompt_map.GENDER.get(o.gender, "person")
    age = prompt_map.AGE_GROUP.get(o.age, "")
    who = " ".join(p for p in [eth, gen, age] if p)
    traits = STYLE_SENT.get(o.face_style) or prompt_map.FACE_STYLE.get(o.face_style, "")
    expr = prompt_map.EXPRESSION.get(o.expression, "")
    mk = prompt_map.MAKEUP.get(o.makeup, "")
    s = f"Replace the face with a different {who}"
    if traits:
        s += f" who has {traits}"
    s += "."
    if expr:
        s += f" Expression: {expr}."
    if mk:
        s += f" Makeup: {mk}."
    s += " Keep the same head pose, gaze direction and lighting. Photorealistic, natural skin."
    return s + KEEP


FINAL_SET = ["salon_01_long_wave_brown", "normal_09_dark_skin", "salon_04_long_wave_black", "normal_01_short_dark"]

for name in FINAL_SET:
    gender, age = N_TARGETS[name]
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, (key, style) in zip(axes[1:], STYLES.items()):
        t = time.time()
        fin, raw, _ = run_gpt_final_v6(src, sentence_prompt_v3(opts_full(gender, age, style)))
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v6_final.png")
        raw.save(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
        ax.imshow(fin); ax.set_title(key, fontsize=12)
        print(f"{short(name)}  {key}  {time.time() - t:.0f}s")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_v6_final.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
name, key = "salon_01_long_wave_brown", "cat"   # 선이 보이는 사진·종으로 바꿔서
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)
raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
fin = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_final.png")
full = src.copy()
full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (lab, im) in zip(axes, [("src", src), ("raw pasted", full), ("final", fin)]):
    ax.imshow(im.crop(jaw)); ax.set_title(lab, fontsize=12); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_v6_jawcheck.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def finalize_v6(src, raw, alpha=0.3, face_dilate=0.10):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm, hm0 = masks.build_face_mask(src), masks.build_hair_mask(src, dilate=0)
    fm_d = dilate_mask(fm, int(fw * face_dilate))
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm_d, hm0), alpha=alpha)
    return compose.recompose_with_hair(src, full, fm_d, hm0)


name, key = "salon_01_long_wave_brown", "cat"
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
jaw = (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3)
raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")

FD = [0.06, 0.10, 0.12]
fig, axes = plt.subplots(1, len(FD) + 1, figsize=(6 * (len(FD) + 1), 7))
axes[0].imshow(src.crop(jaw)); axes[0].set_title("src", fontsize=12)
for ax, d in zip(axes[1:], FD):
    fin = finalize_v6(src, raw, face_dilate=d)
    fin.save(OUT5 / f"c5_{name}_{key}_gpt_v6_fd{d}.png")
    ax.imshow(fin.crop(jaw)); ax.set_title(f"face dilate {d}", fontsize=12)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_v6_facedilate.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def erode_mask(mask, px):
    if px <= 0:
        return mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (px * 2 + 1, px * 2 + 1))
    return Image.fromarray(cv2.erode(np.array(mask.convert("L")), k))


def finalize_v7(src, raw, alpha=0.3, face_dilate=0.10, hair_erode=0.0):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm = masks.build_face_mask(src)
    hm0 = erode_mask(masks.build_hair_mask(src, dilate=0), int(fw * hair_erode))
    fm_d = dilate_mask(fm, int(fw * face_dilate))
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm_d, hm0), alpha=alpha)
    return compose.recompose_with_hair(src, full, fm_d, hm0)


VARS = [("alpha 0.3 / hair 0", 0.3, 0.0), ("alpha 0 / hair 0", 0.0, 0.0),
        ("alpha 0.3 / hair erode 2%", 0.3, 0.02), ("alpha 0.3 / hair erode 4%", 0.3, 0.04)]
fig, axes = plt.subplots(1, len(VARS) + 1, figsize=(6 * (len(VARS) + 1), 7))
axes[0].imshow(src.crop(jaw)); axes[0].set_title("src", fontsize=12)
for ax, (lab, a, e) in zip(axes[1:], VARS):
    fin = finalize_v7(src, raw, alpha=a, hair_erode=e)
    fin.save(OUT5 / f"c5_{name}_{key}_gpt_v7_a{a}_e{e}.png")
    ax.imshow(fin.crop(jaw)); ax.set_title(lab, fontsize=12)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_v7_isolate.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
HAIR_ERODE = 0.04

for name in FINAL_SET:
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, key in zip(axes[1:], STYLES):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
        fin = finalize_v7(src, raw, hair_erode=HAIR_ERODE)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v7_final.png")
        ax.imshow(fin); ax.set_title(key, fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_v7_final.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
name, key = "normal_09_dark_skin", "puppy"
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
hair = (x1 - fw // 4, max(0, y1 - fw // 2), x2 + fw // 4, y1 + fw // 2)  # 이마·앞머리 위주
raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
full = src.copy()
full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))

fig, axes = plt.subplots(1, 4, figsize=(24, 7))
for ax, (lab, im) in zip(axes, [("src", src), ("raw pasted", full),
                                 ("erode 0", finalize_v7(src, raw, hair_erode=0.0)),
                                 ("erode 4%", finalize_v7(src, raw, hair_erode=0.04))]):
    ax.imshow(im.crop(hair)); ax.set_title(lab, fontsize=12); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_v7_haircheck.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def close_mask(mask, px):
    if px <= 0:
        return mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (px * 2 + 1, px * 2 + 1))
    return Image.fromarray(cv2.morphologyEx(np.array(mask.convert("L")), cv2.MORPH_CLOSE, k))


def finalize_v8(src, raw, alpha=0.3, face_dilate=0.10, hair_close=0.077, hair_erode=0.0):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fm = masks.build_face_mask(src)
    hm = masks.build_hair_mask(src, dilate=0)
    hm = close_mask(hm, int(fw * hair_close))
    hm = erode_mask(hm, int(fw * hair_erode))
    fm_d = dilate_mask(fm, int(fw * face_dilate))
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm_d, hm), alpha=alpha)
    return compose.recompose_with_hair(src, full, fm_d, hm)


VARS8 = [("close 0 / erode 0", 0.0, 0.0), ("close .077 / erode 0", 0.077, 0.0),
         ("close .077 / erode 2%", 0.077, 0.02), ("close .077 / erode 4%", 0.077, 0.04)]
CASES = [("normal_09_dark_skin", "puppy", "hair"), ("salon_01_long_wave_brown", "cat", "jaw")]

fig, axes = plt.subplots(len(CASES), len(VARS8) + 1, figsize=(6 * (len(VARS8) + 1), 7 * len(CASES)))
for i, (name, key, region) in enumerate(CASES):
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    crop = ((x1 - fw // 4, max(0, y1 - fw // 2), x2 + fw // 4, y1 + fw // 2) if region == "hair"
            else (x1 - fw // 4, y1 + fw // 3, x2 + fw // 4, y2 + fw // 3))
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
    axes[i, 0].imshow(src.crop(crop)); axes[i, 0].set_title(f"{short(name)} src", fontsize=12)
    for ax, (lab, c, e) in zip(axes[i, 1:], VARS8):
        fin = finalize_v8(src, raw, hair_close=c, hair_erode=e)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v8_c{c}_e{e}.png")
        ax.imshow(fin.crop(crop)); ax.set_title(lab, fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / "grid_v8_close_erode.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
name, key = "normal_09_dark_skin", "puppy"
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
hair_crop = (x1 - fw // 4, max(0, y1 - fw // 2), x2 + fw // 4, y1 + fw // 2)
raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
fin = finalize_v8(src, raw, hair_close=0.077, hair_erode=0.0)

hm = close_mask(masks.build_hair_mask(src, dilate=0), int(fw * 0.077))
fm_d = dilate_mask(masks.build_face_mask(src), int(fw * 0.10))
diff = np.abs(np.array(src).astype(np.int16) - np.array(fin).astype(np.int16)).max(axis=2)
changed = diff > 8

ov = np.array(src).copy()
ov[changed] = (ov[changed] * 0.4 + np.array([255, 0, 0]) * 0.6).astype(np.uint8)
for m, color in [(hm, (0, 255, 0)), (fm_d, (0, 128, 255))]:
    cnts, _ = cv2.findContours(np.array(m), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(ov, cnts, -1, color, 3)

hm_np = np.array(hm) > 127
print(f"바뀐 픽셀 중 헤어 마스크 안: {(changed & hm_np).sum()}  밖: {(changed & ~hm_np).sum()}")

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (lab, im) in zip(axes, [("src", src), ("final", fin), ("changed=red, hair=green, face+10%=blue", Image.fromarray(ov))]):
    ax.imshow(im.crop(hair_crop)); ax.set_title(lab, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_v8_diffmap.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
HAIR_CLOSE, HAIR_ERODE = 0.077, 0.02

for name in FINAL_SET:
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, key in zip(axes[1:], STYLES):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
        fin = finalize_v8(src, raw, hair_close=HAIR_CLOSE, hair_erode=HAIR_ERODE)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v8_final.png")
        ax.imshow(fin); ax.set_title(key, fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_v8_final.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
name = "normal_09_dark_skin"
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
brow = (x1 - fw // 8, y1 + fw // 6, x2 + fw // 8, y1 + fw // 2)
hm = erode_mask(close_mask(masks.build_hair_mask(src, dilate=0), int(fw * 0.077)), int(fw * 0.02))

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for i, key in enumerate(["puppy", "fox"]):
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    fin = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v8_final.png")
    ov = np.array(fin).copy()
    cnts, _ = cv2.findContours(np.array(hm), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(ov, cnts, -1, (0, 255, 0), 2)
    for ax, (lab, im) in zip(axes[i], [("src", src), (f"{key} raw pasted", full), (f"{key} final", fin), ("final + hair mask", Image.fromarray(ov))]):
        ax.imshow(im.crop(brow)); ax.set_title(lab, fontsize=11); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_v8_browcheck.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
def hair_mask_v9(src, fw, hair_close=0.077, hair_erode=0.02, brow_dilate=0.03):
    hm = erode_mask(close_mask(masks.build_hair_mask(src, dilate=0), int(fw * hair_close)), int(fw * hair_erode))
    brow = masks.build_brow_mask(src)
    if brow is None:
        return hm
    h = np.array(hm).copy()
    h[np.array(dilate_mask(brow, int(fw * brow_dilate))) > 127] = 0
    return Image.fromarray(h)


def finalize_v9(src, raw, alpha=0.3, face_dilate=0.10, brow_dilate=0.03):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    hm = hair_mask_v9(src, fw, brow_dilate=brow_dilate)
    fm_d = dilate_mask(masks.build_face_mask(src), int(fw * face_dilate))
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    if alpha > 0:
        full = compose.color_transfer(full, src, masks.build_gen_mask(fm_d, hm), alpha=alpha)
    return compose.recompose_with_hair(src, full, fm_d, hm)


name, key = "normal_09_dark_skin", "puppy"
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
brow = (x1 - fw // 8, y1 + fw // 6, x2 + fw // 8, y1 + fw // 2)
raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")

BD = [0.0, 0.03, 0.05]
fig, axes = plt.subplots(1, len(BD) + 1, figsize=(6 * (len(BD) + 1), 5))
axes[0].imshow(src.crop(brow)); axes[0].set_title("src", fontsize=12)
for ax, d in zip(axes[1:], BD):
    fin = finalize_v9(src, raw, brow_dilate=d)
    fin.save(OUT5 / f"c5_{name}_{key}_gpt_v9_bd{d}.png")
    ax.imshow(fin.crop(brow)); ax.set_title(f"brow dilate {d}", fontsize=12)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_v9_browdilate.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
lb = (x1 + fw // 2, y1 + fw // 5, x2 + fw // 10, y1 + fw * 2 // 5)  # 우리 기준 왼쪽 눈썹
full = src.copy()
full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
ims = [("src", src), ("raw pasted", full)] + [(f"brow dilate {d}", Image.open(OUT5 / f"c5_{name}_{key}_gpt_v9_bd{d}.png")) for d in BD]
fig, axes = plt.subplots(1, len(ims), figsize=(6 * len(ims), 6))
for ax, (lab, im) in zip(axes, ims):
    ax.imshow(im.crop(lb)); ax.set_title(lab, fontsize=12); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_{key}_v9_leftbrow.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
name = "normal_09_dark_skin"
gender, age = N_TARGETS[name]
src = srcs5[name]
x1, y1, x2, y2 = face_box(src)
fw = x2 - x1
brow = (x1 - fw // 8, y1 + fw // 6, x2 + fw // 8, y1 + fw // 2)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, (key, style) in enumerate([("puppy", "강아지상"), ("fox", "여우상")]):
    axes[i, 0].imshow(src.crop(brow)); axes[i, 0].set_title(f"src ({key})", fontsize=12)
    for j in range(2):
        _, raw, _ = run_gpt_final_v6(src, sentence_prompt_v3(opts_full(gender, age, style)))
        raw.save(OUT5 / f"c5_{name}_{key}_gpt_v9_raw_r{j}.png")
        fin = finalize_v9(src, raw, brow_dilate=0.03)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v9_final_r{j}.png")
        axes[i, j + 1].imshow(fin.crop(brow)); axes[i, j + 1].set_title(f"{key} regen {j}", fontsize=12)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT5 / f"grid_{name}_v9_regen_brow.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
OVERRIDE = {("normal_09_dark_skin", "puppy"): "gpt_v9_raw_r0", ("normal_09_dark_skin", "fox"): "gpt_v9_raw_r0"}

for name in FINAL_SET:
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, key in zip(axes[1:], STYLES):
        tag = OVERRIDE.get((name, key), "gpt_v6_raw")
        raw = Image.open(OUT5 / f"c5_{name}_{key}_{tag}.png")
        fin = finalize_v9(src, raw, brow_dilate=0.03)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v9_final.png")
        ax.imshow(fin); ax.set_title(key, fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_v9_final.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
S_ADD = {
    "salon_02_long_wave_dark": ("여성", "20대"),
    "salon_03_long_wave_ring": ("여성", "20대"),
    "salon_05_short_bob_brown": ("여성", "20대"),
}
for name in S_ADD:
    srcs5[name] = storage.to_stored_size(Image.open(SALON / f"{name}.jpg").convert("RGB"))
N_TARGETS.update(S_ADD)

SALON_SET = ["salon_01_long_wave_brown", "salon_02_long_wave_dark", "salon_03_long_wave_ring",
             "salon_04_long_wave_black", "salon_05_short_bob_brown"]

for name in S_ADD:
    gender, age = N_TARGETS[name]
    src = srcs5[name]
    for key, style in STYLES.items():
        t = time.time()
        _, raw, _ = run_gpt_final_v6(src, sentence_prompt_v3(opts_full(gender, age, style)))
        raw.save(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
        print(f"{short(name)}  {key}  {time.time() - t:.0f}s")

for name in SALON_SET:
    src = srcs5[name]
    fig, axes = plt.subplots(1, len(STYLES) + 1, figsize=(6 * (len(STYLES) + 1), 7))
    axes[0].imshow(src); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, key in zip(axes[1:], STYLES):
        raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
        fin = finalize_v9(src, raw, brow_dilate=0.03)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v9_final.png")
        ax.imshow(fin); ax.set_title(key, fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_gpt_v9_final.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
def color_transfer_lab(src_img, ref, mask, alpha_l, alpha_ab):
    s = cv2.cvtColor(np.array(src_img.convert("RGB")), cv2.COLOR_RGB2LAB).astype(np.float32)
    r = cv2.cvtColor(np.array(ref.resize(src_img.size).convert("RGB")), cv2.COLOR_RGB2LAB).astype(np.float32)
    mk = np.array(mask.resize(src_img.size)) > 127
    out = s.copy()
    for c, a in enumerate([alpha_l, alpha_ab, alpha_ab]):
        s_std, s_mean = s[..., c][mk].std(), s[..., c][mk].mean()
        r_std, r_mean = r[..., c][mk].std(), r[..., c][mk].mean()
        t_std, t_mean = s_std + (r_std - s_std) * a, s_mean + (r_mean - s_mean) * a
        if s_std > 1e-6:
            out[..., c] = (s[..., c] - s_mean) / s_std * t_std + t_mean
    return Image.fromarray(cv2.cvtColor(np.clip(out, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB))


def finalize_v10(src, raw, alpha_l=0.3, alpha_ab=0.3, face_dilate=0.10, brow_dilate=0.03):
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    hm = hair_mask_v9(src, fw, brow_dilate=brow_dilate)
    fm_d = dilate_mask(masks.build_face_mask(src), int(fw * face_dilate))
    full = src.copy()
    full.paste(raw.resize((x2 - x1, y2 - y1), Image.LANCZOS), (x1, y1))
    full = color_transfer_lab(full, src, masks.build_gen_mask(fm_d, hm), alpha_l, alpha_ab)
    return compose.recompose_with_hair(src, full, fm_d, hm)


AL = [(0.3, 0.3), (0.6, 0.3), (1.0, 0.3), (0.6, 0.6)]
for name, key in [("salon_01_long_wave_brown", "puppy"), ("salon_03_long_wave_ring", "cat")]:
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fc = (x1 - fw // 4, y1 - fw // 4, x2 + fw // 4, y2 + fw // 2)
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
    fig, axes = plt.subplots(1, len(AL) + 1, figsize=(6 * (len(AL) + 1), 7))
    axes[0].imshow(src.crop(fc)); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, (al, ab) in zip(axes[1:], AL):
        fin = finalize_v10(src, raw, alpha_l=al, alpha_ab=ab)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v10_L{al}_ab{ab}.png")
        ax.imshow(fin.crop(fc)); ax.set_title(f"L {al} / ab {ab}", fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_{key}_v10_alpha_lab.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
for name, key in [("salon_02_long_wave_dark", "puppy"), ("salon_04_long_wave_black", "puppy"), ("salon_05_short_bob_brown", "puppy")]:
    src = srcs5[name]
    x1, y1, x2, y2 = face_box(src)
    fw = x2 - x1
    fc = (x1 - fw // 4, y1 - fw // 4, x2 + fw // 4, y2 + fw // 2)
    raw = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v6_raw.png")
    fig, axes = plt.subplots(1, len(AL) + 1, figsize=(6 * (len(AL) + 1), 7))
    axes[0].imshow(src.crop(fc)); axes[0].set_title(f"{short(name)} src", fontsize=12)
    for ax, (al, ab) in zip(axes[1:], AL):
        fin = finalize_v10(src, raw, alpha_l=al, alpha_ab=ab)
        fin.save(OUT5 / f"c5_{name}_{key}_gpt_v10_L{al}_ab{ab}.png")
        ax.imshow(fin.crop(fc)); ax.set_title(f"L {al} / ab {ab}", fontsize=12)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(OUT5 / f"grid_{name}_{key}_v10_alpha_lab.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
for name, key in [("salon_01_long_wave_brown", "puppy"), ("salon_03_long_wave_ring", "cat"), ("salon_05_short_bob_brown", "puppy")]:
    src = srcs5[name]
    fm = np.array(masks.build_face_mask(src)) > 127
    neck = np.array(masks.build_body_skin_mask(src)) > 127
    def L(img, m): return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)[..., 0][m].mean()
    print(f"{short(name)}  src face {L(src, fm):.1f}  neck {L(src, neck):.1f}")
    for al, ab in AL:
        fin = Image.open(OUT5 / f"c5_{name}_{key}_gpt_v10_L{al}_ab{ab}.png")
        print(f"   L {al}/ab {ab}: face {L(fin, fm):.1f}  neck {L(fin, neck):.1f}")